<a href="https://colab.research.google.com/github/Utkarsh-Jaiswal-code/RAG-Based-HR-assistant/blob/main/ai_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet sentence-transformers==2.2.2
!pip install --quiet transformers==4.33.2
!pip install --quiet huggingface_hub==0.17.3
!pip install --quiet langchain-huggingface==0.1.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.0 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
!pip install langchain-community
!pip install pypdf
from langchain_community.document_loaders import PyPDFLoader
# Load the PDF
loader = PyPDFLoader("/content/Synise_Handbook.pdf")
pages = loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(pages)

In [ ]:
!pip install chromadb
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create or load Chroma vector store
db = Chroma.from_documents(docs, embedding_model, persist_directory="chromadb")
db.persist()  # Save to disk

In [ ]:
retriever = db.as_retriever()
docs = retriever.get_relevant_documents("How many hours of work are required to qualify for group health insurance?")
for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---\n{doc.page_content[:400]}")

In [ ]:
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from transformers import pipeline

falcon_pipeline = pipeline(
    "text-generation",
    model="tiiuae/falcon-7b-instruct",
    max_new_tokens=300,
    temperature=0.1,
    repetition_penalty=1.2,
    trust_remote_code=True,
    device=0,  # use CPU if GPU not available
)

llm = HuggingFacePipeline(pipeline=falcon_pipeline)

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an intelligent HR assistant strictly answering based only on the context below.

If the answer is not explicitly found in the context, respond with "I don't know".DO NOT make up any information.

### Context:
{context}

### Question:
{question}

### Answer:
"""
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": custom_prompt}
)

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
import numpy as np

def answer_query(question):
    try:
        retrieved_docs = retriever.get_relevant_documents(question)

        # Filter 1: No documents found
        if not retrieved_docs:
            return "I don't know. This information is not available in the uploaded document."

        # Filter 2: Short irrelevant documents
        if all(len(doc.page_content.strip()) < 30 for doc in retrieved_docs):
            return "This is sounding like a vague question.Please elaborate it more !"

        # Optional semantic similarity filtering
        embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        query_emb = embedding_model.embed_query(question)

        def cosine_similarity(a, b):
            return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

        doc_scores = [cosine_similarity(query_emb, embedding_model.embed_query(doc.page_content)) for doc in retrieved_docs]

        if max(doc_scores) < 0.55:
            return "This is sounding like a vague question.Please elaborate it more !"

        # Run model if content seems relevant
        return qa.run(question).strip()

    except Exception as e:
        return f"Error: {str(e)}"


In [ ]:
!pip install gradio --quiet
import gradio as gr

# ⚡ Ultra-minimal Gradio UI
def simple_ui():
    gr.Interface(
        fn=answer_query,
        inputs=gr.Textbox(label="Ask a Question", placeholder="E.g., What is the sick leave policy?", lines=2),
        outputs=gr.Textbox(label="Answer"),
        title="📘 HR Assistant",
        description="Ask questions based on the uploaded Employee Handbook PDF.",
        allow_flagging="never",
        live=False,
    ).queue().launch(share=True)

simple_ui()